# Electrical Grid Stability: Linear Regression & Gradient Descent Optimisation

**Dataset:** Electrical Grid Stability (UCI) — 10,000 instances, 12 features, continuous stability target.

**Objective:** Build a linear regression model to predict grid stability, then systematically compare three optimisation approaches.

**Methods compared:**
- Closed-form direct solution (matrix inversion)
- Full-batch gradient descent
- Mini-batch gradient descent (batch size sweep: 1 → 128)
- Learning rate sweep (α = 0.001 → 0.2) on optimal batch size

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time

from scipy.linalg import inv
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

## 1. Dataset & Preparation

**Dataset:** [Electrical Grid Stability](https://archive.ics.uci.edu/dataset/471/electrical+grid+stability+simulated+data) — 10,000 simulated instances, 12 features describing power grid status.

**Target:** `stab` — a continuous value representing grid stability. (`stabf`, the binary threshold indicator, is excluded.)

**Reference:** Schäfer et al., *Taming instabilities in power grid networks by decentralized control*, EPJ Special Topics, 2016.

In [ ]:
df = pd.read_csv(
    "data/electrical_grid_stability_simulated_data.csv",
    skipinitialspace=True
)
df.head()

In [ ]:
from sklearn.model_selection import train_test_split

# Prepare the data
X = df.drop(columns=["stab", "stabf"])  # Features
y = df["stab"]  # Target variable

# Split the data
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=111)

# Standardize the data manually
mean = X_train.mean()
std = X_train.std()

X_train_standardized = (X_train - mean) / std
X_val_standardized = (X_val - mean) / std

# Add a column of 1s
X_train_standardized.insert(0, "bias", 1)
X_val_standardized.insert(0, "bias", 1)

# Verify the results
print("Training data (first 5 rows):")
print(X_train_standardized.head())
print("Validation data (first 5 rows):")
print(X_val_standardized.head())

Training data (first 5 rows):
      bias      tau1      tau2      tau3      tau4        p1        p2  \
27       1 -1.660528  0.430695  0.377521 -1.626030 -1.104238  0.140792   
3847     1  1.661669  0.157388  0.907843 -0.057686 -1.233902  1.424940   
7461     1  0.566266 -1.409389 -0.765733 -0.022768  1.619815 -0.657975   
1356     1  0.817639  1.390610 -0.220468  1.042713 -0.253103  1.444321   
4314     1  1.465225 -0.803014 -1.677004  0.085506  0.294031 -0.758831   

            p3        p4        g1        g2        g3        g4  
27    1.294721  0.485867 -0.183690 -1.699643 -1.112068  0.444101  
3847  0.437191  0.280368  0.793566 -1.204491  0.272084 -0.450099  
7461 -1.229423 -0.930051  1.400131 -1.373953  0.370590 -1.736786  
1356  0.390866 -1.402009  1.424291  0.754523  0.272588  1.577516  
4314 -1.404023  1.656764  0.327629 -0.970964  0.294281 -0.945189  
Validation data (first 5 rows):
      bias      tau1      tau2      tau3      tau4        p1        p2  \
207      1 -1.524

**Why insert the bias column after standardisation, not before?**

The bias column (all 1s) is a constant — mean = 1, std = 0. Standardising it would require dividing by zero, producing undefined values. More fundamentally, the bias term should remain fixed at 1 so the model can learn a constant offset; standardising it destroys that property.

## 2. Direct Solution

Closed-form linear regression: $\mathbf{w} = (X^T X)^{-1} X^T y$. Uses `scipy.linalg.inv` for numerical stability.

In [ ]:
# Convert training and validation data to numpy arrays
X_train_np = X_train_standardized.to_numpy()
y_train_np = y_train.to_numpy()

# Compute weights using the direct solution
weights = inv(X_train_np.T @ X_train_np) @ X_train_np.T @ y_train_np

# Predictions for training and validation sets
y_train_pred = X_train_np @ weights
X_val_np = X_val_standardized.to_numpy()
y_val_pred = X_val_np @ weights

# Calculate RMSE
rmse_train = np.sqrt(mean_squared_error(y_train_np, y_train_pred))
rmse_val = np.sqrt(mean_squared_error(y_val.to_numpy(), y_val_pred))

# Print the results
print(f"RMSE for Training Set: {rmse_train}")
print(f"RMSE for Validation Set: {rmse_val}")

RMSE for Training Set: 0.02196256294427011
RMSE for Validation Set: 0.02184988283270913


## 3. Full-Batch Gradient Descent

Fixed learning rate α = 0.01. Convergence criterion: validation RMSE ≤ 1.0005 × RMSE_direct_solution.

In [ ]:
# Initialize weights
np.random.seed(1001)
weights = np.random.uniform(low=-0.00001, high=0.00001, size=X_train_np.shape[1])

# Learning rate and convergence threshold
alpha = 0.01
convergence_threshold = rmse_val * 1.0005

# Track RMSE
training_rmse = []
validation_rmse = []

# Start timing
start_time = time.time()

# Gradient Descent
converged = False
epoch = 0
while not converged:
    # Predictions
    y_train_pred = X_train_np @ weights

    # Compute gradient
    gradient = -(2 / X_train_np.shape[0]) * (X_train_np.T @ (y_train_np - y_train_pred))

    # Update weights
    weights -= alpha * gradient

    # Compute RMSE
    rmse_train = np.sqrt(mean_squared_error(y_train_np, y_train_pred))
    y_val_pred = X_val_np @ weights
    rmse_val_gd = np.sqrt(mean_squared_error(y_val.to_numpy(), y_val_pred))

    # Store RMSE
    training_rmse.append(rmse_train)
    validation_rmse.append(rmse_val_gd)

    # Check for convergence
    if rmse_val_gd <= convergence_threshold:
        converged = True

    epoch += 1

# End timing
end_time = time.time()

# Print results
print(f"--- Total Training Time: {end_time - start_time:.2f} seconds ---")
print(f"Epochs until convergence: {epoch}")
print(f"Final Validation RMSE: {rmse_val_gd}")

# Plot RMSE vs Epoch
plt.figure(figsize=(10, 6))
plt.plot(range(epoch), training_rmse, label="Training RMSE")
plt.plot(range(epoch), validation_rmse, label="Validation RMSE")
plt.xlabel("Epoch")
plt.ylabel("RMSE")
plt.title("Training and Validation RMSE vs. Epoch")
plt.legend()
plt.grid()
plt.show()

**Overfitting / Underfitting Assessment**

Training and validation RMSE decrease together and converge to nearly identical values. No significant gap at any point — the model generalises well with no sign of overfitting or underfitting.

**Epoch vs Iteration**

- **Epoch:** one full pass through the entire training dataset
- **Iteration:** one weight update step

In full-batch GD, epoch = iteration. In mini-batch/SGD, multiple iterations occur per epoch (one per batch), so epochs × batches_per_epoch = total iterations.

## 4. Mini-Batch & Stochastic Gradient Descent

### 4.1 Implementation

Generalised `mini_batch_gd()` function: accepts any batch size (1 = SGD, N = full-batch). Shuffles training data after each epoch. Stops early if validation RMSE diverges.

In [ ]:
def mini_batch_gd(X_train, y_train, X_val, y_val, batch_size, learning_rate, convergence_threshold):
    np.random.seed(1001)

    # Initialize weights with small random values
    weights = np.random.uniform(low=-0.00001, high=0.00001, size=X_train.shape[1])

    # Initialize arrays to store results
    training_rmse = []
    validation_rmse = []
    elapsed_times = []

    start_time = time.time()
    epoch = 0
    converged = False

    while not converged:
        # Shuffle data
        perm = np.random.permutation(X_train.shape[0])
        X_train_shuffled = X_train[perm]
        y_train_shuffled = y_train[perm]

        # Mini-batch gradient descent
        for i in range(0, X_train.shape[0], batch_size):
            X_batch = X_train_shuffled[i:i + batch_size]
            y_batch = y_train_shuffled[i:i + batch_size]

            # Compute predictions and gradient
            y_batch_pred = X_batch @ weights
            gradient = -(2 / X_batch.shape[0]) * (X_batch.T @ (y_batch - y_batch_pred))

            # Update weights
            weights -= learning_rate * gradient

        # Compute RMSE
        y_train_pred = X_train @ weights
        rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
        y_val_pred = X_val @ weights
        rmse_val = np.sqrt(mean_squared_error(y_val, y_val_pred))

        # Store RMSE and elapsed time
        training_rmse.append(rmse_train)
        validation_rmse.append(rmse_val)
        elapsed_times.append(time.time() - start_time)

        # Check for convergence
        if rmse_val <= convergence_threshold:
            converged = True

        # Stop training if validation RMSE diverges
        if epoch > 0 and validation_rmse[-1] > validation_rmse[-2] * 1.1:
            print("Stopping due to divergence!")
            break

        epoch += 1

    return weights, training_rmse, validation_rmse, elapsed_times

### 4.2 Batch Size Sweep

Sweep batch sizes [1, 2, 4, 8, 16, 32, 64, 128] at α = 0.01.

In [ ]:
# Define parameters
batch_sizes = [1, 2, 4, 8, 16, 32, 64, 128]  # Test different batch sizes
learning_rate = 0.01
convergence_threshold = rmse_val * 1.0005

# Store results
results = {}

# Run for each batch size
for batch_size in batch_sizes:
    print(f"Running Mini-Batch GD with Batch Size {batch_size}...")
    try:
        weights, train_rmse, val_rmse, times = mini_batch_gd(
            X_train_np, y_train_np, X_val_np, y_val.to_numpy(),
            batch_size, learning_rate, convergence_threshold
        )
        results[batch_size] = {
            "train_rmse": train_rmse,
            "val_rmse": val_rmse,
            "times": times,
            "epochs": len(train_rmse),
            "total_time": times[-1] if times else None
        }
    except Exception as e:
        print(f"Batch size {batch_size} failed to converge: {e}")

In [ ]:
# Extract and display the fastest converging batch size
convergence_times = {batch_size: res['total_time'] for batch_size, res in results.items() if res['total_time'] is not None}
fastest_batch_size = min(convergence_times, key=convergence_times.get)

print("Convergence Times for Each Batch Size:", convergence_times)
print(f"Fastest Converging Batch Size: {fastest_batch_size} with Time: {convergence_times[fastest_batch_size]:.2f} seconds")

In [ ]:
plt.figure(figsize=(12, 8))

for batch_size, res in results.items():
    epochs = range(len(res["train_rmse"]))
    plt.plot(epochs, res["train_rmse"], label=f"Train RMSE (Batch {batch_size})", linestyle='--')
    plt.plot(epochs, res["val_rmse"], label=f"Validation RMSE (Batch {batch_size})")

plt.xlabel("Epoch")
plt.ylabel("RMSE")
plt.title("Training and Validation RMSE vs. Epoch for Different Batch Sizes")
plt.legend()
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))

for batch_size, res in results.items():
    times = res["times"]
    plt.plot(times, res["train_rmse"], label=f"Train RMSE (Batch {batch_size})", linestyle='--')
    plt.plot(times, res["val_rmse"], label=f"Validation RMSE (Batch {batch_size})")

plt.xlabel("Time (s)")
plt.ylabel("RMSE")
plt.title("Training and Validation RMSE vs. Time for Different Batch Sizes")
plt.legend()
plt.grid()
plt.show()

In [ ]:
batch_sizes = []
total_times = []

for batch_size, res in results.items():
    if res["total_time"] is not None:
        batch_sizes.append(batch_size)
        total_times.append(res["total_time"])

plt.figure(figsize=(12, 6))
plt.plot(batch_sizes, total_times, marker='o')
plt.xlabel("Batch Size")
plt.ylabel("Total Training Time (s)")
plt.title("Total Training Time vs. Batch Size")
plt.grid()
plt.show()

### 4.3 Findings

- **Larger batches converge faster in wall-clock time** (batch 32: ~0.007 s; batch 2: ~265 s)
- Smaller batches need more epochs but each update is cheaper — the overhead of frequent small updates dominates at batch size 1
- Batch size 1 (SGD) diverged at α = 0.01; noisy gradient estimates prevent stable convergence at this learning rate

## 5. Learning Rate Analysis

### 5.1 Non-Converging Batch Sizes

Batch sizes [1, 4, 8, 16] failed to converge at α = 0.01. Test learning rates [0.001, 0.005, 0.02, 0.05] to find stable configurations.

In [ ]:
# Modified code to include time for convergence
non_converging_batch_sizes = [1, 4, 8, 16]
learning_rates_to_test = [0.001, 0.005, 0.02, 0.05]
non_converging_results = []

for batch_size in non_converging_batch_sizes:
    for lr in learning_rates_to_test:
        try:
            weights, train_rmse, val_rmse, times = mini_batch_gd(
                X_train_np, y_train_np, X_val_np, y_val.to_numpy(),
                batch_size, lr, convergence_threshold
            )
            non_converging_results.append({
                "Batch Size": batch_size,
                "Learning Rate": lr,
                "Training RMSE": train_rmse[-1],
                "Validation RMSE": val_rmse[-1],
                "Convergence Time (s)": times[-1] if times else None,
                "Converged": True
            })
        except Exception:
            non_converging_results.append({
                "Batch Size": batch_size,
                "Learning Rate": lr,
                "Training RMSE": None,
                "Validation RMSE": None,
                "Convergence Time (s)": None,
                "Converged": False
            })

# Create a DataFrame to summarize results
results_df = pd.DataFrame(non_converging_results)

# Display the DataFrame
print(results_df)

**Best configuration from non-converging batch sizes:** batch size 16, α = 0.05 → validation RMSE ≈ 0.02181. Batch size 1 diverged at all tested learning rates.

### 5.2 Learning Rate Sweep

Best batch size from Section 4 (batch 32). Sweep 10 learning rates: 0.001 → 0.2.

In [ ]:
best_batch_size = 32  # Replace with the batch size identified in Part 4
learning_rates = [0.001, 0.002, 0.005, 0.01, 0.02, 0.03, 0.05, 0.07, 0.1, 0.2]

sweeping_results = {}

for lr in learning_rates:
    print(f"Running Mini-Batch GD with Learning Rate {lr}...")
    weights, train_rmse, val_rmse, times = mini_batch_gd(
        X_train_np, y_train_np, X_val_np, y_val.to_numpy(),
        best_batch_size, lr, convergence_threshold
    )
    sweeping_results[lr] = {
        "train_rmse": train_rmse,
        "val_rmse": val_rmse,
        "times": times
    }

In [ ]:
plt.figure(figsize=(12, 8))

for lr, res in sweeping_results.items():
    epochs = range(len(res["train_rmse"]))
    plt.plot(epochs, res["train_rmse"], label=f"Train RMSE (LR {lr:.3f})", linestyle='--')
    plt.plot(epochs, res["val_rmse"], label=f"Validation RMSE (LR {lr:.3f})")

plt.xlabel("Epoch")
plt.ylabel("RMSE")
plt.title("Training and Validation RMSE vs. Epoch (Learning Rate Sweep)")
plt.legend()
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))

for lr, res in sweeping_results.items():
    times = res["times"]
    plt.plot(times, res["train_rmse"], label=f"Train RMSE (LR {lr:.3f})", linestyle='--')
    plt.plot(times, res["val_rmse"], label=f"Validation RMSE (LR {lr:.3f})")

plt.xlabel("Time (s)")
plt.ylabel("RMSE")
plt.title("Training and Validation RMSE vs. Time (Learning Rate Sweep)")
plt.legend()
plt.grid()
plt.show()

### 5.3 Findings

**RMSE vs Epoch:**
- Small learning rates (α ≤ 0.002) take 700+ epochs to converge — stable but slow
- Moderate rates (α = 0.01–0.05) converge smoothly in ~100 epochs
- α ≥ 0.1 shows oscillations and risks divergence

**RMSE vs Time:**
- α = 0.001 takes >5 s; α = 0.05 converges in <2 s
- Optimal: α = 0.05 — lowest RMSE (~0.0218) in the shortest wall-clock time

**Learning Rate Trade-offs**

| Learning Rate | Convergence | RMSE | Notes |
|---|---|---|---|
| α = 0.001–0.002 | Slow (700+ epochs) | ~0.022 | Stable, safe |
| α = 0.01–0.05 | Fast (~100 epochs) | ~0.022 | Best balance |
| α ≥ 0.1 | Unstable | Variable | Risk of divergence |

**Recommended:** batch size 32, α = 0.05

## Conclusions & Key Findings

| Method | Val RMSE | Notes |
|---|---|---|
| Direct solution | 0.0219 | Baseline; instantaneous |
| Full-batch GD | ~0.0219 | Matches direct solution after convergence |
| Mini-batch (best) | ~0.0218 | Batch 32, α = 0.05, <2 s |

**Key takeaways:**
- The closed-form solution is the accuracy ceiling — GD variants converge to the same RMSE
- Larger batches converge faster in wall-clock time (parallelism) but require more epochs than small batches
- Optimal learning rate (α = 0.05) is 5× faster than the conservative default (α = 0.01) with identical final accuracy
- SGD (batch = 1) is unstable at α = 0.01 and failed to converge at any tested learning rate — noisy gradients prevent stable convergence on this dataset